In [2]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# --- 1. Rotation Matrix Functions ---
def rotation_matrix_x(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[1, 0, 0], [0, c, -s], [0, s, c]])

def rotation_matrix_y(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]])

def rotation_matrix_z(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])

def get_euler_rotation(roll, pitch, yaw):
    return rotation_matrix_z(yaw) @ rotation_matrix_y(pitch) @ rotation_matrix_x(roll)

# --- 2. Create UI Widgets ---
# Mode selector
mode_selector = widgets.RadioButtons(
    options=['Euler Angles (Sliders)', 'Custom Rotation Matrix'],
    value='Euler Angles (Sliders)',
    description='Control Mode:',
    style={'description_width': 'initial'}
)

# Sliders for Euler angles
roll_slider = widgets.FloatSlider(min=-np.pi, max=np.pi, step=0.05, value=0.0, description='Roll (X):')
pitch_slider = widgets.FloatSlider(min=-np.pi, max=np.pi, step=0.05, value=0.0, description='Pitch (Y):')
yaw_slider = widgets.FloatSlider(min=-np.pi, max=np.pi, step=0.05, value=0.0, description='Yaw (Z):')

# Textarea for custom matrix input
matrix_input = widgets.Textarea(
    value='[[1.0, 0.0, 0.0],\n [0.0, 1.0, 0.0],\n [0.0, 0.0, 1.0]]',
    description='Matrix R:',
    disabled=False,
    layout=widgets.Layout(width='300px', height='80px'),
    style={'description_width': 'initial'}
)

apply_btn = widgets.Button(description='Apply Matrix', button_style='success')
out = widgets.Output()

# --- 3. Visualization Logic ---
def render_visualizer(R):
    with out:
        clear_output(wait=True)
        
        fig = plt.figure(figsize=(7, 6))
        ax = fig.add_subplot(111, projection='3d')
        
        ax.set_xlim([-1.5, 1.5])
        ax.set_ylim([-1.5, 1.5])
        ax.set_zlim([-1.5, 1.5])
        ax.set_xlabel('X Axis')
        ax.set_ylabel('Y Axis')
        ax.set_zlabel('Z Axis')
        ax.set_title('3D Rigid Body Rotation Visualizer')

        origin = np.array([0, 0, 0])
        
        # Fixed reference frame (dashed lines)
        ax.quiver(*origin, 1, 0, 0, color='r', linestyle='dashed', alpha=0.5, label='Ref X')
        ax.quiver(*origin, 0, 1, 0, color='g', linestyle='dashed', alpha=0.5, label='Ref Y')
        ax.quiver(*origin, 0, 0, 1, color='b', linestyle='dashed', alpha=0.5, label='Ref Z')

        # Rotatable body frame (solid lines)
        ax.quiver(*origin, R[0,0], R[1,0], R[2,0], color='r', length=1.0, label='Body X')
        ax.quiver(*origin, R[0,1], R[1,1], R[2,1], color='g', length=1.0, label='Body Y')
        ax.quiver(*origin, R[0,2], R[1,2], R[2,2], color='b', length=1.0, label='Body Z')

        ax.legend(loc='upper left')
        plt.show()
        
        # Live Matrix Readout & Validation Note
        print("Active Rotation Matrix (R):")
        print("-" * 35)
        for row in R:
            print(f"[ {row[0]:6.3f}  {row[1]:6.3f}  {row[2]:6.3f} ]")
        print("-" * 35)

# --- 4. Event Handlers ---
def update_from_sliders(change=None):
    if mode_selector.value == 'Euler Angles (Sliders)':
        R = get_euler_rotation(roll_slider.value, pitch_slider.value, yaw_slider.value)
        render_visualizer(R)

def on_apply_clicked(b):
    if mode_selector.value == 'Custom Rotation Matrix':
        try:
            # Parse text input safely into a numpy array
            R = np.array(eval(matrix_input.value))
            if R.shape != (3, 3):
                print("Error: Matrix must be 3x3.")
                return
            render_visualizer(R)
        except Exception as e:
            with out:
                print(f"Invalid matrix format. Please use Python list format, e.g., [[1,0,0],[0,1,0],[0,0,1]]\nError: {e}")

def on_mode_change(change):
    if mode_selector.value == 'Euler Angles (Sliders)':
        slider_box.layout.display = 'flex'
        custom_box.layout.display = 'none'
        update_from_sliders()
    else:
        slider_box.layout.display = 'none'
        custom_box.layout.display = 'flex'
        on_apply_clicked(None)

# Layout grouping
slider_box = widgets.VBox([roll_slider, pitch_slider, yaw_slider])
custom_box = widgets.VBox([matrix_input, apply_btn])
custom_box.layout.display = 'none' # Hidden by default

# Bind observers
roll_slider.observe(update_from_sliders, names='value')
pitch_slider.observe(update_from_sliders, names='value')
yaw_slider.observe(update_from_sliders, names='value')
mode_selector.observe(on_mode_change, names='value')
apply_btn.on_click(on_apply_clicked)

# Display complete UI
display(widgets.VBox([mode_selector, slider_box, custom_box, out]))

# Initial render
update_from_sliders()